# Imports

<a href="https://colab.research.google.com/github/sylvainestebe/european-city-inference/blob/main/docs/tutorials/tutorial_1_decision_making.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd

from eci.decision import (
    _get_belief_preference_gap,
    _get_pref_candidate_gap,
    _sample_choice,
    response_function,
    response_function_pref,
    response_function_random,
)

# Local project imports
from eci.environment import EnvConfig, Environment
from eci.plots import plot_belief_trajectory, plot_preference
from eci.utils import _extract_env_data_vectorized, get_voter_trajectory_data
from eci.voting.plurality import _vote_plurality

# Electoral Decision-Making
This tutorial simulates a decision making process where agents (voters) choose between candidates based on their preference and beliefs.

## Setting up the Environment
The environement is made of agent which can be voter or candidate and they get preferences.
First, we initialize the simulation environment. In this example, we create an environement with 1 Voter and 2 Candidates with 2 distinct preference dimensions.

In [ ]:
config = EnvConfig(
    num_voters=1,
    num_candidates=5,
    num_preferences=1,
    scenario=1,
    num_steps=100,
)
env = Environment(config)

You can inspect the agents and get informations about them: Their preferences, if they are candidate or voter and others attributes that will be use in different scenario.

In [ ]:
env.agents

## Visualizing the Agent
You can visualize the agent's internal model. Bottom Nodes ($x_0): These represent observations per preference.Top Nodes ($x_1, $x_2): These track the mean and the precision of the nodes 0.

In [ ]:
env.network.plot_network()

## Inspecting Preferences
An agent's preference is defined by two parameters: 
**Mean**: The position of the preference
**Precision**: The confidence in that preference.
You can display the preference of your agent, by selecting from the environement which agent you want to plot.

In [ ]:
# Get the preference for one agent (preference = (mean,precision)).
env.voters[0].preferences["mean"]
env.voters[0].preferences["precision"]

We can manually change the preferences to test specific scenarios. Here, we force the Voter to strictly prefer the position [0, 0] (the center) with low precision [0.1, 0.1].

In [ ]:
# Modify the preference.
env.voters[0].preferences["mean"] = jnp.array([0])
env.voters[0].preferences["precision"] = jnp.array([0.1])
env.voters[0]

Then we have to run the inference. This is running the inference about the world given some observation per preference and use all the pyhgf mechanisticary 

In [ ]:
env._run_multi_agent_inference()

# Inspecting Preferences

In [ ]:
# extract the data from the environment
data = _extract_env_data_vectorized(env)

In [ ]:
fig, ax = plot_preference(data)
fig.show()

## Inspecting Beliefs
For each preference, agent get observation about the world and this contribute to form belief about the world on this specific preference.

In [ ]:
traj_data = get_voter_trajectory_data(env, voter_id=0)
# Plot the belief trajectory
fig, ax1, ax2 = plot_belief_trajectory(**traj_data)
tonic_volatility = env.voters[0].tonic_volatility
fig.suptitle(f"Tonic volatility: {tonic_volatility:.2f}")
fig.show()

## The Decision Loop
Now that we have defined the agent's beliefs and preferences. The agent must transform its observations into a decision.

This process involves three mathematical steps, executed at every iteration:

**Calculating the "Gap" (KL Divergence):**
    The agent measures the distance between its *preferences* (what it wants) and its *beliefs* about the world (what it perceives).

In [ ]:
pref_belief_gap = _get_belief_preference_gap(
    data["beliefs"]["mean"],
    data["beliefs"]["precision"],
    data["preferences"]["mean"],
    data["preferences"]["precision"],
)
pref_belief_gap

**Calculating the "Gap" (KL Divergence):**
    The agent measures the distance between its *preferences* (what it wants) and the candidate policy.

In [ ]:
pref_candidate_gap = _get_pref_candidate_gap(
    data["preferences"]["mean"],
    data["preferences"]["precision"],
    data["candidates"]["mean"],
    data["candidates"]["precision"],
)
pref_candidate_gap

Then the agent select the candidate that minimize the disatifaction.

In [ ]:
candidate_utilities = pref_belief_gap[:, jnp.newaxis] - pref_candidate_gap
candidate_utilities

Finally, we randomly select a winner following this probability distribution. This is the final act of voting.

In [ ]:
key = jax.random.PRNGKey(int(time.time()))
# Split the JAX key for two separate random samples
key_round_1, key_round_2 = jax.random.split(key)
# Create mask for round 1 (all candidates are eligible)
mask_round_1 = jnp.ones_like(candidate_utilities, dtype=bool)
masked_utilities = jnp.where(mask_round_1, candidate_utilities, -jnp.inf)

# Sample round 1 vote
vote_1, softmax_probs_1 = _sample_choice(key_round_1, masked_utilities)
vote_1

# Multiple time

We can also do it multiple times

In [ ]:
# Do multiple vote for the same agent.
key = jax.random.PRNGKey(int(time.time()))
list_vote = []
num_votes = 100

for i in range(num_votes):
    key, key_iteration = jax.random.split(key)
    key_round_1, key_round_2 = jax.random.split(key_iteration)
    mask_round_1 = jnp.ones_like(candidate_utilities, dtype=bool)
    masked_utilities = jnp.where(mask_round_1, candidate_utilities, -jnp.inf)
    vote_1, softmax_probs_1 = _sample_choice(key_round_1, masked_utilities)
    list_vote.append(vote_1)
list_vote

The graph above displays the distribution of accumulated votes.
Here we can see in the result that the agent mostly vote for the candidate which reduce his disatifaction.

In [ ]:
vote_counts = pd.Series(list_vote).value_counts()
ax = vote_counts.plot.bar(
    color="teal",
    figsize=(8, 6),
    rot=0,
    title="Vote per Candidates",
    xlabel="Candidates",
    ylabel="Vote",
)
ax.bar_label(ax.containers[0], padding=5)
plt.tight_layout()

There is different way to compute the utility for each canditas, you can explore this in the decision module or create your own

# Different kind of response function

In [ ]:
key = jax.random.PRNGKey(42)
data = _extract_env_data_vectorized(env)

## Random (Baseline)

In [ ]:
# Run simulations
key = jax.random.PRNGKey(int(time.time()))
data = _extract_env_data_vectorized(env)
sim_plurality = env.run_n_simulation(
    func=_vote_plurality,
    data=data,
    response_function=response_function_random,
    key=key,
    n_simulations=100,
)
winners = [r["winner"] for r in sim_plurality.values()]
vote_counts = pd.Series(winners).value_counts()
ax = vote_counts.plot.bar(
    color="teal",
    figsize=(8, 6),
    rot=0,
    title="Vote per Candidates",
    xlabel="Candidates",
    ylabel="Vote",
)
ax.bar_label(ax.containers[0], padding=5)
plt.tight_layout()

## Using only preference

In [ ]:
# Run simulations
key = jax.random.PRNGKey(int(time.time()))
data = _extract_env_data_vectorized(env)
sim_plurality = env.run_n_simulation(
    func=_vote_plurality,
    data=data,
    response_function=response_function_pref,
    key=key,
    n_simulations=100,
)
winners = [r["winner"] for r in sim_plurality.values()]
vote_counts = pd.Series(winners).value_counts()
ax = vote_counts.plot.bar(
    color="teal",
    figsize=(8, 6),
    rot=0,
    title="Vote per Candidates",
    xlabel="Candidates",
    ylabel="Vote",
)
ax.bar_label(ax.containers[0], padding=5)
plt.tight_layout()

## Using preference and belief

In [ ]:
# Run simulations
key = jax.random.PRNGKey(int(time.time()))
data = _extract_env_data_vectorized(env)
sim_plurality = env.run_n_simulation(
    func=_vote_plurality,
    data=data,
    response_function=response_function,
    key=key,
    n_simulations=100,
)
winners = [r["winner"] for r in sim_plurality.values()]
vote_counts = pd.Series(winners).value_counts()
ax = vote_counts.plot.bar(
    color="teal",
    figsize=(8, 6),
    rot=0,
    title="Vote per Candidates",
    xlabel="Candidates",
    ylabel="Vote",
)
ax.bar_label(ax.containers[0], padding=5)
plt.tight_layout()